# 09. Phase 2F YOLO Retraining on Verified Expanded Dataset
## AI Driver Monitoring & Drowsiness Detection System
### Transfer Learning, Training Dynamics & Checkpoint Export

**Project:** AI Driver Monitoring & Drowsiness Detection System  
**Phase:** Phase 2F — YOLO Retraining, Benchmarking & Generalization Evaluation  
**Author:** Antigravity AI Research & Engineering Team  
**Date:** October 2026  
**Status:** COMPLETE & EMPIRICALLY VERIFIED (Transfer Learning on 512 Images, Model C Exported)

---

### Core Principles & Governance Rules:
1. **Canonical Schema:** 
   - `0 = eyes_closed`
   - `1 = eyes_open`
   - `2 = yawning`
2. **Dataset Integrity:** Trained on Phase 2E verified dataset (256 original + 256 train-only augmented frames = 512 training pool). Validation (44 frames) and Test (33 frames) remain **100% unaugmented and untouched**.
3. **Model Family:** Lightweight `YOLOv5nu` (anchor-free P5 decoupled head, ~2.5M parameters, 7.1 GFLOPs) fine-tuned via transfer learning from official COCO weights.
4. **Checkpoint Preservation:** Baseline anchor `best.pt` and Phase 2D checkpoint `weights/improved_best.pt` are frozen. The retrained model is saved as `weights/phase2f_best.pt`.


---
## 1. Environment Setup & Configuration
Initialize paths, deterministic seeds (`seed=42`), and verify environment compatibility.


In [1]:
import os
import sys
import csv
import json
import random
from pathlib import Path
import cv2
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path("..").resolve() if Path("..").resolve().name == "Drowsiness-Detection-using-YOLOv5" else Path(".").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "annotated"
CONFIG_PATH = PROJECT_ROOT / "configs" / "yolo_phase2f.yaml"
WEIGHTS_DIR = PROJECT_ROOT / "weights"

print(f"Project Root: {PROJECT_ROOT}")
print(f"Dataset YAML: {CONFIG_PATH} (Exists: {CONFIG_PATH.exists()})")
print(f"Weights Dir:  {WEIGHTS_DIR}")


Project Root: C:\Users\msiva\Music\Drowsiness-Detection-using-YOLOv5
Dataset YAML: C:\Users\msiva\Music\Drowsiness-Detection-using-YOLOv5\configs\yolo_phase2f.yaml (Exists: True)
Weights Dir:  C:\Users\msiva\Music\Drowsiness-Detection-using-YOLOv5\weights


---
## 2. Dataset Ingestion & Partition Verification
Verify the Phase 2E dataset split counts prior to training.


In [2]:
train_imgs = list((DATA_DIR / "images" / "train").glob("*.jpg"))
val_imgs = list((DATA_DIR / "images" / "val").glob("*.jpg"))
test_imgs = list((DATA_DIR / "images" / "test").glob("*.jpg"))

print(f"Training Pool:   {len(train_imgs)} images (256 original + 256 augmented)")
print(f"Validation Set:  {len(val_imgs)} images (100% unaugmented held-out subjects)")
print(f"Test Set:        {len(test_imgs)} images (100% unaugmented held-out subjects)")

assert len(train_imgs) == 512, f"Expected 512 train images, got {len(train_imgs)}"
assert len(val_imgs) == 44, f"Expected 44 val images, got {len(val_imgs)}"
assert len(test_imgs) == 33, f"Expected 33 test images, got {len(test_imgs)}"
print("ASSERTION PASSED: Exact Phase 2E dataset partitions verified.")


Training Pool:   512 images (256 original + 256 augmented)
Validation Set:  44 images (100% unaugmented held-out subjects)
Test Set:        33 images (100% unaugmented held-out subjects)
ASSERTION PASSED: Exact Phase 2E dataset partitions verified.


---
## 3. Training Configuration & Hyperparameter Architecture
Configure training hyperparameters calibrated for edge CPU/GPU transfer learning.


In [3]:
train_hyperparams = {
    "model_family": "YOLOv5nu (Anchor-Free P5 Decoupled Head)",
    "pretrained_weights": "yolov5nu.pt (Official COCO Checkpoint)",
    "input_resolution": "416 x 416",
    "epochs": 15,
    "batch_size": 16,
    "optimizer": "AdamW",
    "initial_lr (lr0)": 0.002,
    "final_lr_factor (lrf)": 0.01,
    "momentum": 0.937,
    "weight_decay": 0.0005,
    "warmup_epochs": 1.0,
    "loss_gains": {"box": 7.5, "cls": 0.5, "dfl": 1.5},
    "augmentation": {"mosaic": 0.2, "fliplr": 0.5, "mixup": 0.0},
    "hardware": "Intel Core i7-10700 CPU @ 2.90GHz (16 threads)",
    "software": "Python 3.13.13, PyTorch 2.14.1+cpu, Ultralytics 8.4.171"
}
print(json.dumps(train_hyperparams, indent=2))


{
  "model_family": "YOLOv5nu (Anchor-Free P5 Decoupled Head)",
  "pretrained_weights": "yolov5nu.pt (Official COCO Checkpoint)",
  "input_resolution": "416 x 416",
  "epochs": 15,
  "batch_size": 16,
  "optimizer": "AdamW",
  "initial_lr (lr0)": 0.002,
  "final_lr_factor (lrf)": 0.01,
  "momentum": 0.937,
  "weight_decay": 0.0005,
  "warmup_epochs": 1.0,
  "loss_gains": {
    "box": 7.5,
    "cls": 0.5,
    "dfl": 1.5
  },
  "augmentation": {
    "mosaic": 0.2,
    "fliplr": 0.5,
    "mixup": 0.0
  },
  "hardware": "Intel Core i7-10700 CPU @ 2.90GHz (16 threads)",
  "software": "Python 3.13.13, PyTorch 2.14.1+cpu, Ultralytics 8.4.171"
}


---
## 4. Training Loss Convergence & Metric Progression
Inspect the training progression logged across all 15 epochs in `runs/phase2f/train/results.csv`.


In [4]:
results_csv = PROJECT_ROOT / "runs" / "phase2f" / "train" / "results.csv"
if results_csv.exists():
    with open(results_csv, "r", encoding="utf-8") as f:
        rows = list(csv.DictReader(f))
    print(f"Logged Epochs in results.csv: {len(rows)}")
    print(f"{'Epoch':<6} | {'Train Box':<10} | {'Train Cls':<10} | {'Train DFL':<10} | {'Val mAP50':<10} | {'Val mAP50-95'}")
    print("-" * 65)
    for r in rows:
        ep = int(r["epoch"])
        if ep in [1, 3, 5, 8, 10, 12, 15]:
            print(f"{ep:<6} | {float(r['train/box_loss']):<10.4f} | {float(r['train/cls_loss']):<10.4f} | {float(r['train/dfl_loss']):<10.4f} | {float(r['metrics/mAP50(B)']):<10.4f} | {float(r['metrics/mAP50-95(B)']):.4f}")
else:
    print(f"Results CSV not found at {results_csv}")


Logged Epochs in results.csv: 15
Epoch  | Train Box  | Train Cls  | Train DFL  | Val mAP50  | Val mAP50-95
-----------------------------------------------------------------
1      | 0.7605     | 1.8416     | 1.0375     | 0.4899     | 0.3690
3      | 0.5295     | 0.8844     | 0.9069     | 0.9950     | 0.8691
5      | 0.4429     | 0.6818     | 0.8847     | 0.9950     | 0.8914
8      | 0.3114     | 0.4305     | 0.8271     | 0.9950     | 0.9175
10     | 0.2556     | 0.3895     | 0.8018     | 0.9379     | 0.9209
12     | 0.2274     | 0.3248     | 0.7817     | 0.9950     | 0.9271
15     | 0.2032     | 0.2803     | 0.7841     | 0.9950     | 0.9267


---
## 5. Visualizing Training Loss & Validation Metrics
Plot the loss convergence and mAP progression across training epochs.


In [5]:
if results_csv.exists():
    epochs = [int(r["epoch"]) for r in rows]
    box_loss = [float(r["train/box_loss"]) for r in rows]
    cls_loss = [float(r["train/cls_loss"]) for r in rows]
    dfl_loss = [float(r["train/dfl_loss"]) for r in rows]
    val_map50 = [float(r["metrics/mAP50(B)"]) for r in rows]
    val_map50_95 = [float(r["metrics/mAP50-95(B)"]) for r in rows]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(epochs, box_loss, label="Box Loss", marker="o", color="#1f77b4")
    axes[0].plot(epochs, cls_loss, label="Classification Loss", marker="s", color="#ff7f0e")
    axes[0].plot(epochs, dfl_loss, label="DFL Loss", marker="^", color="#2ca02c")
    axes[0].set_xlabel("Epoch", fontsize=11)
    axes[0].set_ylabel("Loss", fontsize=11)
    axes[0].set_title("Training Loss Convergence", fontsize=12, fontweight="bold")
    axes[0].legend()
    axes[0].grid(True, linestyle="--", alpha=0.5)
    
    axes[1].plot(epochs, val_map50, label="Val mAP@0.5", marker="o", color="#d62728", lw=2)
    axes[1].plot(epochs, val_map50_95, label="Val mAP@0.5:0.95", marker="s", color="#9467bd", lw=2)
    axes[1].set_xlabel("Epoch", fontsize=11)
    axes[1].set_ylabel("Validation mAP", fontsize=11)
    axes[1].set_title("Validation mAP Progression", fontsize=12, fontweight="bold")
    axes[1].legend()
    axes[1].grid(True, linestyle="--", alpha=0.5)
    
    plt.tight_layout()
    out_fig = PROJECT_ROOT / "results" / "phase2f" / "phase2f_training_curves.png"
    out_fig.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_fig, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Training curves saved to {out_fig.name}")


Training curves saved to phase2f_training_curves.png


---
## 6. Checkpoint Export & Preservation Verification
Confirm that the new model is saved as `weights/phase2f_best.pt` and earlier checkpoints are preserved.


In [6]:
chk_base = PROJECT_ROOT / "best.pt"
chk_p2d = PROJECT_ROOT / "weights" / "improved_best.pt"
chk_p2f = PROJECT_ROOT / "weights" / "phase2f_best.pt"

print("Model Checkpoint Audit:")
print(f"  Model A (Baseline best.pt):             Exists = {chk_base.exists()} | Size = {chk_base.stat().st_size / (1024*1024):.2f} MB")
print(f"  Model B (Phase 2D improved_best.pt):     Exists = {chk_p2d.exists()} | Size = {chk_p2d.stat().st_size / (1024*1024):.2f} MB")
print(f"  Model C (Phase 2F phase2f_best.pt):      Exists = {chk_p2f.exists()} | Size = {chk_p2f.stat().st_size / (1024*1024):.2f} MB")

assert chk_base.exists(), "Model A missing!"
assert chk_p2d.exists(), "Model B missing!"
assert chk_p2f.exists(), "Model C missing!"
print("ASSERTION PASSED: All three model checkpoints verified and independently accessible.")


Model Checkpoint Audit:
  Model A (Baseline best.pt):             Exists = True | Size = 5.01 MB
  Model B (Phase 2D improved_best.pt):     Exists = True | Size = 4.98 MB
  Model C (Phase 2F phase2f_best.pt):      Exists = True | Size = 4.99 MB
ASSERTION PASSED: All three model checkpoints verified and independently accessible.
